In [6]:
import pandas as pd
import plotly.graph_objects as go
import numpy as np

# --- 1. Load and Prepare Data ---

# Define file paths for the two simulation runs
g2_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_multiple_1_4_2_2_0.seq_1024.batch_128/run_20251202_161612_918ms/g2/T5_Base_multiple_1_4_2_2_0.seq_1024.batch_128_trace_matched_timing.csv'
ns3_filepath = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_multiple_1_4_2_2_0.seq_1024.batch_128/run_20251202_161614_005ms/ns3/T5_Base_multiple_1_4_2_2_0.seq_1024.batch_128_trace_matched_timing.csv'

# Load the CSV files into pandas DataFrames
try:
    g2_df = pd.read_csv(g2_filepath)
    ns3_df = pd.read_csv(ns3_filepath)
    print("✅ Successfully loaded both g2 and ns3 data files.")
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}. Please ensure the file paths are correct.")
    # Stop execution if files are not found
    raise

# Filter for communication nodes only (node_type 4 is for computation)
comm_node_types = [5, 6, 7]
g2_comm = g2_df[g2_df['node_type'].isin(comm_node_types)].copy()
ns3_comm = ns3_df[ns3_df['node_type'].isin(comm_node_types)].copy()
print(f"Filtered for communication nodes. Found {len(g2_comm)} comm nodes in g2, {len(ns3_comm)} in ns3.")

# --- 2. Merge DataFrames for Comparison ---

# Select and rename columns for a clean merge
# We use 'node_name' and 'sys_id' as a composite key to uniquely identify an operation on a specific GPU
g2_subset = g2_comm[['sys_id', 'node_name', 'issue_tick', 'callback_tick', 'elapsed_time']]
ns3_subset = ns3_comm[['sys_id', 'node_name', 'issue_tick', 'callback_tick', 'elapsed_time']]

# Rename columns to distinguish between g2 and ns3 after merging
g2_subset = g2_subset.rename(columns={
    'issue_tick': 'g2_issue_tick',
    'callback_tick': 'g2_callback_tick',
    'elapsed_time': 'g2_elapsed_time'
})
ns3_subset = ns3_subset.rename(columns={
    'issue_tick': 'ns3_issue_tick',
    'callback_tick': 'ns3_callback_tick',
    'elapsed_time': 'ns3_elapsed_time'
})

# Merge the two dataframes on the GPU ID and the operation name
merged_df = pd.merge(g2_subset, ns3_subset, on=['sys_id', 'node_name'], how='inner')

# --- 3. Visualize Callback Tick Divergence for Each GPU ---

# Sort the data chronologically based on the g2 simulation's start time
# This gives us the intended order of operations
merged_df_sorted = merged_df.sort_values(by=['sys_id', 'g2_issue_tick']).reset_index(drop=True)

print("\n--- Plotting Callback Tick Comparison for Each NPU ---")
print("Generating a plot for each NPU to visualize the divergence in operation completion times ('callback_tick').\n")

# Get the list of unique GPUs (sys_id)
gpus = merged_df_sorted['sys_id'].unique()
gpus.sort()

# --- 4. Parse Workload Groups and Assign Markers ---
import re
import os
import sys

# Ensure we can import from the root
sys.path.append('/app/astra-sim')

from upc.generate_workloads_split import create_collectives_log

# Define the workload folder (containing the original ET files)
workload_folder = '/app/astra-sim/upc/comparing_networks/workload/T5_Base_multiple_1_4_2_2_0.seq_1024.batch_128'

# Path to the log file
workload_log_path = os.path.join(workload_folder, 'duplicate_collectives.log')

# Generate the log if it doesn't exist
if not os.path.exists(workload_log_path):
    create_collectives_log(workload_folder)

# Define a list of markers to cycle through
# Full list: circle, square, diamond, cross, x, triangle-up, triangle-down, etc.
markers = ['circle', 'square', 'diamond', 'cross', 'x', 'star', 'hexagram', 'triangle-up', 'triangle-down', 'pentagon']
workload_to_marker = {}
workload_to_signature = {}
collective_to_workload = {}

try:
    with open(workload_log_path, 'r') as f:
        lines = f.readlines()
        i = 0
        while i < len(lines):
            line = lines[i]
            workload_match = re.match(r'^Workload: (\S+)', line)
            if workload_match:
                current_workload = workload_match.group(1)
                if current_workload not in workload_to_marker:
                    marker_index = len(workload_to_marker) % len(markers)
                    workload_to_marker[current_workload] = markers[marker_index]
                
                # Look for signature on the next line
                if (i + 1 < len(lines)) and (signature_match := re.match(r'^\s+Signature: (.*)', lines[i+1])):
                    workload_to_signature[current_workload] = signature_match.group(1).strip()
            
            collective_match = re.match(r'^\s+-\s(.+)', line)
            if collective_match and current_workload:
                collective_name = collective_match.group(1).strip()
                collective_to_workload[collective_name] = current_workload
            i += 1
    print(f"✅ Successfully parsed {len(workload_to_marker)} workload groups from log file.")
except FileNotFoundError:
    print(f"❌ Warning: Workload log file not found at {workload_log_path}. Using default markers.")
    collective_to_workload = {} # Ensure it's empty if file not found

# Create a mapping from node_name to marker symbol
default_marker = 'circle'
merged_df_sorted['marker'] = merged_df_sorted['node_name'].apply(
    lambda name: workload_to_marker.get(collective_to_workload.get(name), default_marker)
)

# Generate a plot for each GPU
from plotly.subplots import make_subplots
import math

rows = 4
cols = 4
fig = make_subplots(
    rows=rows, 
    cols=cols, 
    subplot_titles=[f'NPU {gpu_id}' for gpu_id in gpus[:rows*cols]],
    shared_xaxes=True,
    vertical_spacing=0.05
)


for i, gpu_id in enumerate(gpus):
    if i >= rows * cols:
        print(f"Warning: Only plotting first {rows*cols} of {len(gpus)} GPUs.")
        break

    # Calculate subplot position
    row = (i // cols) + 1
    col = (i % cols) + 1

    # Filter the dataframe for the current GPU
    gpu_df = merged_df_sorted[merged_df_sorted['sys_id'] == gpu_id].copy()
    
    # The y-axis will represent the order of communication operations
    operation_order = np.arange(len(gpu_df))
    
    # To plot all segments efficiently, we build lists with 'None' to break the lines
    g2_x, g2_y, g2_hover, g2_markers = [], [], [], []
    ns3_x, ns3_y, ns3_hover, ns3_markers = [], [], [], []

    for j, op_idx in enumerate(operation_order):
        node_name = gpu_df['node_name'].iloc[j]
        marker_symbol = gpu_df['marker'].iloc[j]
        
        # Extend g2 lists
        g2_x.extend([gpu_df['g2_issue_tick'].iloc[j], gpu_df['g2_callback_tick'].iloc[j], None])
        g2_y.extend([op_idx, op_idx, None])
        g2_hover.extend([node_name, node_name, None])
        g2_markers.extend([marker_symbol, marker_symbol, marker_symbol])

        # Extend ns3 lists
        ns3_x.extend([gpu_df['ns3_issue_tick'].iloc[j], gpu_df['ns3_callback_tick'].iloc[j], None])
        ns3_y.extend([op_idx, op_idx, None])
        ns3_hover.extend([node_name, node_name, None])
        ns3_markers.extend([marker_symbol, marker_symbol, marker_symbol])

    # Add traces for g2 and ns3 to the subplot
    fig.add_trace(go.Scatter(
        x=g2_x, y=g2_y,
        mode='lines+markers',
        name='g2',
        line=dict(color='blue'),
        marker=dict(size=6, symbol=g2_markers),
        hovertext=g2_hover,
        hovertemplate='<b>%{hovertext}</b><br>Time: %{x}<br>Order: %{y}<extra></extra>',
        legendgroup='g2',
        showlegend=(i==0)
    ), row=row, col=col)
    
    fig.add_trace(go.Scatter(
        x=ns3_x, y=ns3_y,
        mode='lines+markers',
        name='ns3',
        line=dict(color='orange'),
        marker=dict(size=6, symbol=ns3_markers),
        hovertext=ns3_hover,
        hovertemplate='<b>%{hovertext}</b><br>Time: %{x}<br>Order: %{y}<extra></extra>',
        legendgroup='ns3',
        showlegend=(i==0)
    ), row=row, col=col)

# --- Add Dummy Traces for Marker Legend ---
# Add invisible traces to the first subplot just to create the legend
for workload_name, marker in workload_to_marker.items():
    signature = workload_to_signature.get(workload_name, "N/A")
    fig.add_trace(go.Scatter(
        x=[None], y=[None], # No data
        mode='markers',
        marker=dict(symbol=marker, color='black', size=8),
        name=f"{workload_name}<br>  └─ Signature: {signature}",
        legendgroup='Workloads',
        showlegend=True
    ), row=1, col=1)


# Update layout for the entire grid
fig.update_layout(
    title_text='Operation Duration (Issue to Callback) for All NPUs',
    height=1600,
    width=2000,
    legend_title="Simulation & Workloads",
    legend=dict(
        tracegroupgap=20 # Add space between 'g2'/'ns3' and the workload markers
    )
)

# Update axis titles for the grid
fig.update_xaxes(title_text='Time (ticks)')
fig.update_yaxes(title_text='Comm. Op. Order')
    
fig.show()

print(f"✅ Generated plots for all {len(gpus)} NPUs.")


✅ Successfully loaded both g2 and ns3 data files.
Filtered for communication nodes. Found 544 comm nodes in g2, 544 in ns3.

--- Plotting Callback Tick Comparison for Each NPU ---
Generating a plot for each NPU to visualize the divergence in operation completion times ('callback_tick').

✅ Successfully parsed 12 workload groups from log file.


✅ Generated plots for all 16 NPUs.


In [11]:
merged_df_sorted[(merged_df_sorted['sys_id'] == 14)].sort_values(by='g2_issue_tick')

,sys_id,node_name,g2_issue_tick,g2_callback_tick,g2_elapsed_time,ns3_issue_tick,ns3_callback_tick,ns3_elapsed_time
1932,14,shadow_mb0.transformer.11.ffn_res.y@0_Y_RECV,0,34480954160618,34480954160618,0,58896235505536,58896235505536
1933,14,mb0.transformer.12.mha.qkv@0_X1COMM,34480954320877,34676080959464,195126638587,58896235665795,59726524045181,830288379386
1934,14,mb0.transformer.12.mha.dwqkv@0_X2_COMM,34676080959464,35195939299734,519858340270,59726524045181,60006848402765,280324357584
1935,14,mb0.transformer.12.mha.o@0_X1COMM,35195939299734,35391066419099,195127119365,60006848402765,60504167789327,497319386562
1936,14,mb0.transformer.12.ffn.x00@0_X1COMM,35391066739617,35910924278596,519857538979,60504168109845,61193544877903,689376768058
...,...,...,...,...,...,...,...,...
2065,14,mb0.transformer.12.ffn.dx0@0_X1COMM,185629939083809,185629957352478,18268669,154176444474380,154176444955177,480797
2066,14,mb0.transformer.12.mha.do1@0_X1COMM,185629957672996,187051904812455,1421947139459,154176445275695,155087148296903,910703021208
2067,14,mb0.transformer.12._sharded_grad@0_X1COMM,187051911653232,188441217889856,1389306236624,155087155137680,155666798360773,579643223093
2068,14,mb0.transformer.12.mha.dx@0_X1COMM,188441217889856,188441218370653,480797,155666798360773,156462643468963,795845108190
